- https://python.langchain.com/v0.1/docs/modules/tools/custom_tools/
  - handling tool exception

In [1]:
import numpy as np
import re
import polars as pl
import scipy

from langchain.callbacks.manager import CallbackManagerForRetrieverRun
from langchain.chains import LLMChain
from langchain_chroma import Chroma
from langchain.document_loaders import JSONLoader
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.llms import Ollama
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.documents.base import Document
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence  # , RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)



In [2]:
path = "/mnt/d/temp/user/ed/mart/mmplastic/20240725_154000_000000/data.json"

In [3]:
# question = "어떤 대학교가 미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구하는지 찾아줘"
question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구와 관련된 미국 대학교를 찾고있어"
# question = "미세플라스틱이 햇빛에 노출되면 오염줄질을 흡수하는 연구를 한 대학 어디야?"

In [4]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            # [f"{d.metadata['seq_num']}-{d.metadata['sub_seq_num']} RANK:{i+1}\n{d.metadata['title'][:20]}:\n\n" + d.page_content for i, d in enumerate(docs)]
            [f"{d.metadata['seq_num']} RANK:{i+1}\n{d.metadata['title'][:64]}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["title"] = record.get("title")
    metadata["date"] = record.get("date")
    return metadata

loader = JSONLoader(
    file_path=path,
    jq_schema=".[]",
    content_key="content",
    text_content=True,
    metadata_func=metadata_func
)
text_splitter = RecursiveCharacterTextSplitter(
    separators="\n\n",
    chunk_size=200,
    chunk_overlap=0,
    keep_separator=True
)
embeddings = OllamaEmbeddings(
    model="tiger-gemma2"
)

In [5]:
llm = ChatOllama(
    model="tiger-gemma2",
    temperature=0.8,
    num_predict=320,
)

response = llm.invoke(question)
print(f"[{response.response_metadata['eval_duration'] / np.power(10., 9)} sec.]:\n" + "" + response.content)


[2.46686 sec.]:
죄송합니다만, 저는 미세플라스틱이 햇빛에 노출되었을 때 오염줄질을 흡수하는 연구와 관련된 미국 대학을 알려드릴 수 없습니다. 저는 대부분의 질문에 대한 답변을 할 수 있도록 설계되었지만, 이번에는 도움이 되지 못하겠습니다.


In [6]:
import yaml
from langchain.agents import create_json_agent
from langchain_community.agent_toolkits import JsonToolkit
from langchain_community.tools.json.tool import JsonSpec
from langchain_openai import OpenAI

In [7]:
# with open("openai_openapi.yml") as f:
#     data = yaml.load(f, Loader=yaml.FullLoader)
data = {"academy":[{"university": "Arizona State University"}, {"university": "California State University"}]}
json_spec = JsonSpec(dict_=data, max_value_length=4000)
json_toolkit = JsonToolkit(spec=json_spec)

json_agent_executor = create_json_agent(
    llm=llm, toolkit=json_toolkit, verbose=True
)

In [8]:
try:
    json_agent_executor.invoke("What universities are in academy")
except Exception as ope:
    logging.error(str(ope))



> Entering new AgentExecutor chain...
Action: json_spec_list_keys
Action Input: data
Observation: ['academy']
Thought:

ERROR:root:An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: ``


I can use those keys to help me find the answer. Let's start with 'academy'.

What universities are in academy? First, we will want to see what's inside this key, so let's call `json_spec_list_keys` again but with the path "data['academy']". This should give us a list of keys that exist at that level.

Action: json_spec_list_keys
Action Input: data["academy"]
Observation: ValueError('Value at path `data["academy"]` is not a dict, get the value directly.')
Thought:

In [9]:
from langchain.agents import initialize_agent
from langchain.tools import DuckDuckGoSearchRun
from langchain.agents import Tool
from langchain.tools import BaseTool
from langchain.chains.conversation.memory import ConversationBufferWindowMemory 

search = DuckDuckGoSearchRun()

def meaning_of_life(input=""):
    return '삶은 달걀'

search_tool = Tool(
    name = 'search',
    func = search.run,
    description="useful for when you need to answer questions about current events. You should ask targeted questions"
    # description="current event에 대해 도움이 되는 답을 해라. 목표로 하는 질문을 해야한다."
)
life_tool = Tool(
    name='삶이란 무엇인가?',
    func= meaning_of_life,
    description="Useful for when you need to answer questions about the meaning of life. input should be MOL "
)

tools = [search_tool, life_tool]

memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=3,
    return_messages=True
)
conversational_agent = initialize_agent(
    agent = 'chat-conversational-react-description',
    tools = tools,
    llm = llm,
    verbose = True,
    max_iterations = 3,
    early_stopping_method = 'generate',
    memory = memory
)
conversational_agent("삶이란 무엇인가?")

/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(
/home/ed/miniconda3/envs/mypy311/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(




> Entering new AgentExecutor chain...


ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: Hey there! It seems like you're curious about life and its meaning. Let me help you out with that!  I can offer some insights on what makes up the essence of life, but first, let's talk about some concepts that will be helpful in understanding this complex topic.

First, we need to understand that life is a state characterized by continuous change and adaptation. Living organisms are constantly interacting with their environment, responding to stimuli, and evolving over time. This dynamic nature of life is driven by the fundamental processes of metabolism, growth, reproduction, and response to external cues.

Now, when it comes to the meaning of life, there's no single answer that will satisfy everyone. Different cultures and belief systems have their own unique interpret

# json chat agent
- https://python.langchain.com/v0.2/docs/integrations/tools/ddg/

In [10]:
from langchain import hub
from langchain.agents import AgentExecutor, create_json_chat_agent

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/react-chat-json")
print(type(prompt))

# prompt = PromptTemplate(
#     input_variables=["input"],
#     template=""" 주어진 질문에 충실하게 한글로 답변해줘
#     질문: {input}
#     """
# )

tools = [DuckDuckGoSearchRun(max_results=4, verbose=True)]
agent = create_json_chat_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

# agent_executor.invoke({"input": "what is LangChain?"})
agent_executor.invoke({"input": "대한민국 미세플라스틱 연구 권위자"})

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>


> Entering new AgentExecutor chain...
Could not parse LLM output: Hey there! So you want to know about experts in microplastics research in South Korea? Let me give you the scoop. 

Microplastics are tiny plastic particles that are found all over the place, including our oceans, lakes, and even the air we breathe. In South Korea, there's been a growing concern about these pollutants and how they can affect human health and the environment. This has led to more research in this field, with many researchers dedicated to studying microplastics and their impact on South Korea.

Now, when it comes to identifying the top experts in this area, it can be tricky. There isn't one single definitive list that everyone agrees on. It really depends on what kind of work you're looking for - environmental science, toxicology, or even policy analysis related to microplastics.

But hey, I've got your back! Based on my research and knowledge abou

{'input': '대한민국 미세플라스틱 연구 권위자',
 'output': 'There are three prominent researchers in microplastic research in South Korea that I can tell you about: Professor Jae-Hyun Kim from Incheon National University, Dr. Ji-Hye Park from Yonsei University, and Professor Yong-Kyu Lim from Sogang University. Each of these experts has made significant contributions to the field.'}

# Humma-as-a-tool
- https://python.langchain.com/v0.2/docs/integrations/tools/human_tools/
- https://python.langchain.com/v0.1/docs/modules/tools/custom_tools/
- https://python.langchain.com/v0.2/docs/how_to/custom_tools/

# Custom tool
- https://api.python.langchain.com/en/latest/llms/langchain_community.llms.gradient_ai.GradientLLM.html

In [11]:
# StructuredTool.from_function(
#     func=,
#     name=,
#     description=,
#     args_schema=,
#     handle_tool_error=,
# )

In [37]:
import time
import uuid
from langchain_core.tools import BaseTool
import random
from typing import Any, Dict, List, Tuple

from langchain_core.tools import tool

from pydantic import BaseModel as PydBaseModel
from langchain.pydantic_v1 import BaseModel, Field
from typing_extensions import TypedDict


class ToolCallInput(PydBaseModel):
    type: str = "tool_call"
    id: str
    args: dict

# class ToolCallRequest(BaseModel):
#     type: str = Field(description="tool_call", default="tool_call")
#     id: str = Field(description="uuid")
#     args: dict = Field(description="args")

# class InputRequest(TypedDict):
class InputRequest(BaseModel):
    min: float = Field(description="min")
    max: float = Field(description="max")
    size: int = Field(description="size")

class InputArgs(PydBaseModel):
    min: float = Field(description="min")
    max: float = Field(description="max")
    size: int = Field(description="size")


class GenerateRandomFloats(BaseTool):
    name: str = "generate_random_floats"
    description: str = "Generate size random floats in the range [min, max]."
    response_format: str = "content_and_artifact"

    ndigits: int = 2

    def forward(self, request: Dict[str, Any]):
        """
        StructuredTool
        """
    # def forward(self, min: float, max: float, size: int):
        # print(f"### {request}")
        # request = InputRequest.parse_obj(request)
        min = request["min"]
        max = request["max"]
        size = request["size"]

        range_ = max - min
        array = [
            round(min + (range_ * random.random()), ndigits=self.ndigits)
            for _ in range(size)
        ]
        content = f"Generated {size} floats in [{min}, {max}], rounded to {self.ndigits} decimals."
        return content, array

    # def _run(self, min: float, max: float, size: int) -> Tuple[str, List[float]]:
    def _run(self, request: ToolCallInput) -> Tuple[str, List[float]]:
        input_args = InputArgs.model_validate(request.args)
        min = input_args.min
        max = input_args.max
        size = input_args.size

        range_ = max - min
        array = [
            round(min + (range_ * random.random()), ndigits=self.ndigits)
            for _ in range(size)
        ]
        content = f"Generated {size} floats in [{min}, {max}], rounded to {self.ndigits} decimals."
        return content, array
        # return content, {"content": content, "values": array}

    def invoke(self, request: InputArgs):
        tool_request = ToolCallInput(
            id=f"{self.name}:{uuid.uuid1()}",
            args=request.model_dump(),
        )
        return super().invoke(tool_request.model_dump())
        # return super().invoke(tool_request)

rand_gen = GenerateRandomFloats(ndigits=4)

In [27]:
s = time.time()
rand_gen.invoke(
    request=InputArgs(min=0.1, max=3.3333, size=3)
)

print(time.time()-s)

0.0007240772247314453


### RunnableLambda(func).as_tools(args_schema)

In [38]:
from langchain_core.runnables import RunnableLambda

custom_tool = RunnableLambda(rand_gen.forward).as_tool(InputRequest)

In [15]:
print(type(custom_tool))

<class 'langchain_core.tools.StructuredTool'>


In [16]:
InputRequest(min=0.1, max=3.3333, size=3)

InputRequest(min=0.1, max=3.3333, size=3)

In [39]:
## slow against custom invoke with pydantic BaseModel Request
s = time.time()
response = custom_tool.invoke(input=InputRequest(min=0.1, max=3.3333, size=3).dict())

print(response)
print(time.time()-s)

KeyError: 'min'

In [18]:
## slow against custom invoke with pydantic BaseModel Request
s = time.time()
custom_tool.invoke(input=InputArgs(min=0.1, max=3.3333, size=3).model_dump())
print(time.time()-s)

0.0020885467529296875


# Wikipedia
- https://python.langchain.com/v0.2/docs/integrations/tools/wikipedia/

# Test with_structured_output

In [19]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

class Person(BaseModel):
    """Personal information"""
    name: str
    
    
# model = ChatOpenAI()
try:
    model = llm.with_structured_output(Person)
    model.invoke('Bob is a person.')
except Exception as re:
    print(f"{type(re).__name__}: {str(re)}")
    """ResponseError: tiger-gemma2 does not support tools"""
    pass

ResponseError: tiger-gemma2 does not support tools
